In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Configuración de semilla para reproducibilidad (Business Consistency)
np.random.seed(42)

def generar_dataset_retail(n_transacciones=10000, n_clientes=800):
    """
    Genera un dataset transaccional simulando un e-commerce real.
    Output: Archivo CSV 'ecommerce_transactions.csv'
    """
    print(f"--- Iniciando simulación de {n_transacciones} transacciones ---")
    
    # 1. SERIES DE TIEMPO (Cronología)
    # Rango: Últimos 12 meses
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365)
    days_range = (end_date - start_date).days
    
    # Generación de fechas aleatorias y ordenamiento cronológico
    random_days = np.random.randint(0, days_range, n_transacciones)
    invoice_dates = [start_date + timedelta(days=int(d)) for d in random_days]
    invoice_dates.sort()

    # 2. COMPORTAMIENTO DE CLIENTES (Principio de Pareto 80/20)
    # Pocos clientes realizan la mayoría de las compras
    customer_ids = np.arange(1000, 1000 + n_clientes)
    # Distribución Dirichlet para pesos desbalanceados
    probabilidades = np.random.dirichlet(np.ones(n_clientes) * 0.5) 
    simulated_customers = np.random.choice(customer_ids, size=n_transacciones, p=probabilidades)

    # 3. PRECIOS Y PRODUCTOS (Distribución Log-Normal)
    # Simula precios reales: muchos productos baratos, pocos productos premium
    unit_prices = np.random.lognormal(mean=2.5, sigma=0.8, size=n_transacciones)
    unit_prices = np.round(unit_prices, 2)
    
    # Cantidades: Mayor probabilidad de comprar 1 o 2 unidades (Retail estándar)
    quantities = np.random.choice(
        [1, 2, 3, 4, 5, 10, 20], 
        size=n_transacciones, 
        p=[0.7, 0.15, 0.05, 0.04, 0.03, 0.02, 0.01]
    )

    # 4. CONSOLIDACIÓN DEL DATAFRAME
    df = pd.DataFrame({
        'InvoiceNo': [f"INV-{10000+i}" for i in range(n_transacciones)],
        'InvoiceDate': invoice_dates,
        'CustomerID': simulated_customers,
        'Quantity': quantities,
        'UnitPrice': unit_prices,
        'Country': np.random.choice(['USA', 'UK', 'Spain', 'Germany', 'France'], size=n_transacciones)
    })

    # KPI Calculado: Total de la línea
    df['TotalAmount'] = df['Quantity'] * df['UnitPrice']
    
    return df

# --- EJECUCIÓN ---
df_ecommerce = generar_dataset_retail()

# Validación de Negocio (Vista previa)
print("\nVista previa del Dataset:")
print(df_ecommerce.head())
print("\nEstadísticas descriptivas:")
print(df_ecommerce[['TotalAmount']].describe())

# Guardado con nombre profesional
nombre_archivo = 'ecommerce_transactions.csv'
df_ecommerce.to_csv(nombre_archivo, index=False)
print(f"\n✅ Archivo '{nombre_archivo}' generado correctamente.")

--- Iniciando simulación de 10000 transacciones ---

Vista previa del Dataset:
   InvoiceNo                InvoiceDate  CustomerID  Quantity  UnitPrice  \
0  INV-10000 2024-12-29 23:40:28.920977        1339         2      10.81   
1  INV-10001 2024-12-29 23:40:28.920977        1027         1       4.99   
2  INV-10002 2024-12-29 23:40:28.920977        1731         1      34.50   
3  INV-10003 2024-12-29 23:40:28.920977        1372         1       7.97   
4  INV-10004 2024-12-29 23:40:28.920977        1010         2       7.01   

   Country  TotalAmount  
0       UK        21.62  
1   France         4.99  
2   France        34.50  
3  Germany         7.97  
4       UK        14.02  

Estadísticas descriptivas:
        TotalAmount
count  10000.000000
mean      30.865960
std       56.484635
min        0.660000
25%        8.595000
50%       15.745000
75%       31.560000
max     1376.000000

✅ Archivo 'ecommerce_transactions.csv' generado correctamente.
